# Метод множителей Лагранжа

In [49]:
import sympy

In [50]:
def gradient(expr, variables):
    return [sympy.diff(expr, v) for v in variables]

In [51]:
def lagrange_min(f, g):
    variables = sorted(list(f.free_symbols.union(g.free_symbols)), key=lambda s: s.name)
    lam = sympy.Symbol('lambda')
    L = f - lam * g
    equations = gradient(L, variables + [lam])
    solutions = sympy.solve(equations, variables + [lam], dict=True)
    candidates = [(tuple(float(s[v]) for v in variables), float(f.subs(s))) for s in solutions]
    
    return min(candidates, key=lambda item: item[1])

In [52]:
x, y = sympy.symbols('x y')
f = x**2 + y**2
g = x - 2 * y - 5
print(lagrange_min(f, g))

((1.0, -2.0), 5.0)


# Метод Зойтендейка

In [53]:
import numpy as np

In [54]:
def f(x):
    return x[0]**2 + x[1]**2

def g(x):
    return x[0] - 2 * x[1] - 5

# def f(x):
#     return x[0] - x[1]

# def g(x):
#     return x[0]**2 - x[1]**2 - 1

In [55]:
def gradient(f, x, h=1e-6):
    n = len(x)
    grad = np.zeros(n)
    for i in range(n):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += h
        x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

In [56]:
def zoutendijk(f, g, x0, max_iter=1000, tol=1e-6):
    x = np.array(x0, dtype=float)
    for _ in range(max_iter):
        gf = gradient(f, x)
        gg = gradient(g, x)
        ng = np.dot(gg, gg)

        if ng < 1e-12:
            break
        d = -gf + (np.dot(gf, gg) / ng) * gg

        if np.linalg.norm(d) < tol:
            break
        alpha = 1.0

        for _ in range(50):
            xn = x + alpha * d
            if f(xn) < f(x):
                break
            alpha *= 0.5

        x = xn
        gx = g(x)
        x = x - (gx / ng) * gg
        if abs(g(x)) < tol and np.linalg.norm(gradient(f, x)) < tol:
            break
        
    return x, f(x)

In [57]:
x0 = [1, 0]
point, value = zoutendijk(f, g, x0)
print(point, value)

[ 1. -2.] 4.999999998881776


# Метод проекции градиента Розена

In [58]:
def golden_section_search(f, a, b, tol=1e-6):
    phi = (1 + np.sqrt(5)) / 2
    resphi = 2 - phi
    
    x1 = a + resphi * (b - a)
    x2 = b - resphi * (b - a)
    f1 = f(x1)
    f2 = f(x2)
    
    while abs(b - a) > tol:
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            x1 = a + resphi * (b - a)
            f1 = f(x1)
        else:
            a = x1
            x1 = x2
            f1 = f2
            x2 = b - resphi * (b - a)
            f2 = f(x2)
    
    return (a + b) / 2

In [59]:
def rosen_gradient_projection(f, g_funcs, g_vals, x0, eps=1e-6, max_iter=1000):
    x = np.array(x0, dtype=float)
    m = len(g_funcs)
    
    for _ in range(max_iter):
        A = np.array([gradient(g, x) for g in g_funcs])
        b = np.array(g_vals, dtype=float) - np.array([g(x) for g in g_funcs])
        
        P = np.eye(len(x)) - A.T @ np.linalg.inv(A @ A.T) @ A
        
        gf = gradient(f, x)
        S = -P @ gf
        
        if np.linalg.norm(S) < eps:
            return x, f(x)
        
        alpha_max = np.inf
        for k in range(m):
            denom = A[k] @ S
            if denom > 0:
                val = (b[k] - A[k] @ x) / denom
                if val < alpha_max:
                    alpha_max = val
            elif denom < 0:
                val = (b[k] - A[k] @ x) / denom
                if val > 0 and val < alpha_max:
                    alpha_max = val
        
        if alpha_max == np.inf:
            alpha_max = 1
        elif alpha_max <= 0:
            alpha_max = 0
        
        def phi(alpha):
            return f(x + alpha * S)
        
        alpha_opt = golden_section_search(phi, 0, alpha_max)
        x = x + alpha_opt * S
    
    return x, f(x)

In [60]:
g_funcs = [g]
g_vals = [0]
x0 = [10, 2.5]

x_min, f_min = rosen_gradient_projection(f, g_funcs, g_vals, x0)

print("Точка минимума:", x_min)
print("Значение целевой функции:", f_min)

Точка минимума: [ 1. -2.]
Значение целевой функции: 4.999999984012785
